In [5]:
import json
from collections import Counter

TRAIN_JSON = r"E:\Projects\Automatic Numerical Fact Verification Using Reasoning Traces\task 2\data\train.json"
VAL_JSON   = r"E:\Projects\Automatic Numerical Fact Verification Using Reasoning Traces\task 2\data\validation.json"
TEST_JSON  = r"E:\Projects\Automatic Numerical Fact Verification Using Reasoning Traces\task 2\data\test.json"


def analyze_dataset(json_path, label_key):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    total_samples = len(data)

    trace_correct_counter = Counter()
    trace_total_counter = Counter()

    samples_with_no_correct = 0
    samples_with_all_correct = 0

    print("=" * 80)
    print(f"Dataset : {json_path}")
    print("=" * 80)

    for sample_idx, sample in enumerate(data):

        gt = sample[label_key]

        traces = sample["Reasoning_traces"]
        verdicts = sample["Verdict_list"]

        correct_indices = []

        for i, pred in enumerate(verdicts):

            trace_total_counter[i] += 1

            if pred == gt:
                trace_correct_counter[i] += 1
                correct_indices.append(i)

        if len(correct_indices) == 0:
            samples_with_no_correct += 1

        if len(correct_indices) == len(verdicts):
            samples_with_all_correct += 1

        # Uncomment if you want every sample printed
        # print(f"Sample {sample_idx}")
        # print("Ground Truth:", gt)
        # print("Correct Trace Indices:", correct_indices)
        # print()

    print(f"Total samples           : {total_samples}")
    print(f"No correct trace        : {samples_with_no_correct}")
    print(f"All traces correct      : {samples_with_all_correct}")

    print("\nPer Trace Accuracy")
    print("-" * 40)

    for idx in sorted(trace_total_counter):
        acc = 100 * trace_correct_counter[idx] / trace_total_counter[idx]
        print(
            f"Trace {idx+1:2d}: "
            f"{trace_correct_counter[idx]:5d}/{trace_total_counter[idx]:5d} "
            f"({acc:.2f}%)"
        )

    return data


# Run analysis
train = analyze_dataset(TRAIN_JSON, "label")
val   = analyze_dataset(VAL_JSON, "label")
test  = analyze_dataset(TEST_JSON, "Label")

JSONDecodeError: Expecting property name enclosed in double quotes: line 111 column 1 (char 82002)

In [2]:
!pip install langdetect

     ---------------------------------------- 0.0/981.5 kB ? eta -:--:--
     ---------- ----------------------------- 262.1/981.5 kB ? eta -:--:--
     ---------------------------------------- 981.5/981.5 kB 3.5 MB/s  0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993362 sha256=efcd79225294083f730030a8f8411f6951719ede0346553d3dee3727bd1cefd7
  Stored in directory: c:\users\hp\appdata\local\pip\cache\wheels\eb\87\25\2dddf1c94e1786054e25022ec5530bfed52bad86d882999c48
Successfully built langdetect


In [6]:
import re
import json
import codecs
from langdetect import detect

def detect_lang_simple(text):
    # fallback heuristic based on unicod|e ranges
    ranges = {
        'zh': [(0x4E00, 0x9FFF), (0x3400, 0x4DBF)],   # CJK
        'ja': [(0x3040, 0x30FF), (0x4E00, 0x9FFF)],   # Hiragana/Katakana + CJK
        'ko': [(0xAC00, 0xD7AF)],                     # Hangul
        'ar': [(0x0600, 0x06FF), (0x0750, 0x077F)],   # Arabic
        'hi': [(0x0900, 0x097F)],                     # Devanagari
        'ru': [(0x0400, 0x04FF)],                     # Cyrillic
        'he': [(0x0590, 0x05FF)],                     # Hebrew
        'el': [(0x0370, 0x03FF)],                     # Greek
        'th': [(0x0E00, 0x0E7F)],                     # Thai
        'la': [(0x0000, 0x007F)]                      # ASCII/Latin (default)
    }
    counts = {k:0 for k in ranges}
    for ch in text:
        cp = ord(ch)
        for k, rs in ranges.items():
            for a,b in rs:
                if a <= cp <= b:
                    counts[k] += 1
    # pick top non-zero, prefer non-latin if present
    items = sorted(counts.items(), key=lambda x: x[1], reverse=True)
    return items[0][0] if items[0][1] > 0 else 'unknown'

# try to use langdetect if available
try:
    def detect_lang(text):
        try:
            return detect(text)
        except:
            return detect_lang_simple(text)
except Exception:
    detect_lang = detect_lang_simple

escaped_pattern_str = re.compile(
    r'"(?P<key>[^"]*(?:[Cc]laim|[Ee]vid)[^"]*)"\s*:\s*(?P<val>"(?:[^"\\]|\\.)*\\u[0-9a-fA-F]{4}(?:[^"\\]|\\.)*")',
    flags=re.S
)
escaped_pattern_arr = re.compile(
    r'"(?P<key>[^"]*(?:[Cc]laim|[Ee]vid)[^"]*)"\s*:\s*(?P<val>\[(?:[^\]]*?\\u[0-9a-fA-F]{4}[^\]]*?)\])',
    flags=re.S
)

def analyze_file_for_escaped(file_path):
    raw = open(file_path, "r", encoding="utf-8").read()
    claims_count = 0
    evidences_count = 0
    texts = []  # decoded texts for language detection

    for m in escaped_pattern_str.finditer(raw):
        key = m.group('key').lower()
        val_raw = m.group('val')  # quoted JSON string with escapes
        inner = val_raw[1:-1]
        # decode JSON-style escapes (\uXXXX, \n, \")
        try:
            decoded = codecs.decode(inner, 'unicode_escape')
        except Exception:
            # fallback: remove backslashes before quotes and backslashes
            decoded = inner.replace('\\"', '"').replace('\\\\', '\\')
        if 'claim' in key:
            claims_count += 1
        else:
            evidences_count += 1
        texts.append(decoded)

    for m in escaped_pattern_arr.finditer(raw):
        key = m.group('key').lower()
        val_raw = m.group('val')  # array substring
        try:
            parsed = json.loads(val_raw)
        except Exception:
            parsed = []
        any_contains = False
        for item in parsed:
            if isinstance(item, str):
                # if the array had \u escapes, this match indicates it; include items
                texts.append(item)
                any_contains = True
        if any_contains:
            if 'claim' in key:
                claims_count += 1
            else:
                evidences_count += 1

    # detect languages for decoded texts
    langs = set()
    for t in texts:
        if not t or t.isspace():
            continue
        langs.add(detect_lang(t))

    return {
        "file": file_path,
        "claims_with_escaped": claims_count,
        "evidences_with_escaped": evidences_count,
        "languages": sorted(langs)
    }

results = {}
for path in (TRAIN_JSON, VAL_JSON, TEST_JSON):
    results[path] = analyze_file_for_escaped(path)

print(results)

{'E:\\Projects\\Automatic Numerical Fact Verification Using Reasoning Traces\\task 2\\data\\train.json': {'file': 'E:\\Projects\\Automatic Numerical Fact Verification Using Reasoning Traces\\task 2\\data\\train.json', 'claims_with_escaped': 1604, 'evidences_with_escaped': 1, 'languages': ['af', 'da', 'de', 'en', 'es', 'fr', 'hi', 'tl']}, 'E:\\Projects\\Automatic Numerical Fact Verification Using Reasoning Traces\\task 2\\data\\validation.json': {'file': 'E:\\Projects\\Automatic Numerical Fact Verification Using Reasoning Traces\\task 2\\data\\validation.json', 'claims_with_escaped': 424, 'evidences_with_escaped': 0, 'languages': ['de', 'en']}, 'E:\\Projects\\Automatic Numerical Fact Verification Using Reasoning Traces\\task 2\\data\\test.json': {'file': 'E:\\Projects\\Automatic Numerical Fact Verification Using Reasoning Traces\\task 2\\data\\test.json', 'claims_with_escaped': 435, 'evidences_with_escaped': 1903, 'languages': ['ca', 'da', 'de', 'en', 'es', 'fr', 'hi', 'id', 'it', 'ko',

In [ ]:
TRAIN_JSON = r"E:\Projects\Automatic Numerical Fact Verification Using Reasoning Traces\task 2\data\train.json"
VAL_JSON   = r"E:\Projects\Automatic Numerical Fact Verification Using Reasoning Traces\task 2\data\validation.json"
TEST_JSON  = r"E:\Projects\Automatic Numerical Fact Verification Using Reasoning Traces\task 2\data\test.json"



In [13]:
import pandas as pd
import json
import codecs
import re

# reuse existing regex patterns if available, else compile (safe to redefine)
try:
    escaped_str_re = escaped_pattern_str
    escaped_arr_re = escaped_pattern_arr
except NameError:
    escaped_str_re = re.compile(
        r'"(?P<key>[^\"]*(?:[Cc]laim|[Ee]vid)[^\"]*)"\s*:\s*(?P<val>"(?:[^\"\\]|\\.)*\\u[0-9a-fA-F]{4}(?:[^\"\\]|\\.)*")',
        flags=re.S
    )
    escaped_arr_re = re.compile(
        r'"(?P<key>[^\"]*(?:[Cc]laim|[Ee]vid)[^\"]*)"\s*:\s*(?P<val>\[(?:[^\]]*?\\u[0-9a-fA-F]{4}[^\]]*?)\])',
        flags=re.S
    )

def extract_escaped_evidences_from_raw(raw):
    escaped_texts = set()
    for m in escaped_str_re.finditer(raw):
        key = m.group('key').lower()
        if 'evid' not in key:
            continue
        inner = m.group('val')[1:-1]
        try:
            decoded = codecs.decode(inner, 'unicode_escape')
        except Exception:
            decoded = inner.replace('\\"', '"').replace('\\\\', '\\')
        escaped_texts.add(decoded)
    for m in escaped_arr_re.finditer(raw):
        key = m.group('key').lower()
        if 'evid' not in key:
            continue
        val_raw = m.group('val')
        try:
            parsed = json.loads(val_raw)
            for item in parsed:
                if isinstance(item, str):
                    escaped_texts.add(item)
        except Exception:
            pass
    return escaped_texts


def analyze_traces_vs_evidence(file_path):
    raw = open(file_path, "r", encoding="utf-8").read()
    escaped_evidence_texts = extract_escaped_evidences_from_raw(raw)

    # load JSON safely: accept both a JSON array or newline-delimited JSON (NDJSON)
    try:
        data = json.loads(raw)  # full JSON document
    except json.JSONDecodeError:
        data = []
        for line in raw.splitlines():
            line = line.strip()
            if not line:
                continue
            try:
                data.append(json.loads(line))
            except Exception:
                # skip lines that still fail to parse
                continue

    # normalize data into list of dicts
    if isinstance(data, dict):
        data = [data]
    elif isinstance(data, list):
        newdata = []
        for item in data:
            if isinstance(item, dict):
                newdata.append(item)
            elif isinstance(item, str):
                try:
                    parsed = json.loads(item)
                    if isinstance(parsed, dict):
                        newdata.append(parsed)
                except Exception:
                    continue
        data = newdata
    else:
        data = []

    rows = []
    total_traces = 0
    literal_escape_traces = 0
    uses_escaped_evidence = 0
    uses_nonescaped_evidence = 0
    uses_both_types = 0
    uses_no_evidence_ref = 0

    for sample in data:
        traces = sample.get("Reasoning_traces", [])
        # normalize traces to list
        if isinstance(traces, str):
            traces = [traces]
        # collect evidences present in this sample (heuristic: keys containing 'evid')
        sample_evid_texts = []
        for k, v in sample.items():
            if 'evid' in k.lower():
                if isinstance(v, list):
                    for it in v:
                        if isinstance(it, str):
                            sample_evid_texts.append(it)
                elif isinstance(v, str):
                    sample_evid_texts.append(v)

        for tr in traces:
            if not isinstance(tr, str):
                continue
            total_traces += 1
            has_literal = '\\u' in tr
            if has_literal:
                literal_escape_traces += 1
            # check evidence references
            esc_ref = False
            nonesc_ref = False
            for ev in sample_evid_texts:
                if not ev:
                    continue
                if ev in tr:
                    if ev in escaped_evidence_texts:
                        esc_ref = True
                    else:
                        nonesc_ref = True
            if esc_ref and nonesc_ref:
                uses_both_types += 1
                uses_escaped_evidence += 1
                uses_nonescaped_evidence += 1
            elif esc_ref:
                uses_escaped_evidence += 1
            elif nonesc_ref:
                uses_nonescaped_evidence += 1
            else:
                uses_no_evidence_ref += 1

    return {
        "file": file_path,
        "total_traces": total_traces,
        "traces_with_literal_u": literal_escape_traces,
        "traces_using_escaped_evidence": uses_escaped_evidence,
        "traces_using_nonescaped_evidence": uses_nonescaped_evidence,
        "traces_using_both_types": uses_both_types,
        "traces_using_no_evidence_ref": uses_no_evidence_ref,
        "escaped_evidence_count_in_raw": len(escaped_evidence_texts)
    }

files = [TRAIN_JSON, VAL_JSON, TEST_JSON]
summary = [analyze_traces_vs_evidence(p) for p in files]
df = pd.DataFrame(summary)
df = df[[
    "file",
    "escaped_evidence_count_in_raw",
    "total_traces",
    "traces_with_literal_u",
    "traces_using_escaped_evidence",
    "traces_using_nonescaped_evidence",
    "traces_using_both_types",
    "traces_using_no_evidence_ref"
]]
display(df)


,file,escaped_evidence_count_in_raw,total_traces,traces_with_literal_u,traces_using_escaped_evidence,traces_using_nonescaped_evidence,traces_using_both_types,traces_using_no_evidence_ref
0,E:\Projects\Automatic Numerical Fact Verificat...,3,0,0,0,0,0,0
1,E:\Projects\Automatic Numerical Fact Verificat...,0,24000,0,0,269,0,23731
2,E:\Projects\Automatic Numerical Fact Verificat...,8780,51160,0,121,47,0,50992


In [14]:
import re
import json
import codecs
import pandas as pd

files = [TRAIN_JSON, VAL_JSON, TEST_JSON]

# helpers

def load_json_flexible(path):
    raw = open(path, 'r', encoding='utf-8').read()
    try:
        data = json.loads(raw)
    except Exception:
        data = []
        for line in raw.splitlines():
            line = line.strip()
            if not line:
                continue
            try:
                data.append(json.loads(line))
            except Exception:
                continue
    # normalize
    if isinstance(data, dict):
        return [data]
    if isinstance(data, list):
        out = []
        for item in data:
            if isinstance(item, dict):
                out.append(item)
            elif isinstance(item, str):
                try:
                    parsed = json.loads(item)
                    if isinstance(parsed, dict):
                        out.append(parsed)
                except Exception:
                    continue
        return out
    return []


def extract_escaped_evidences_from_raw(raw):
    escaped = set()
    str_re = re.compile(
        r'"(?P<key>[^\"]*(?:[Cc]laim|[Ee]vid)[^\"]*)"\s*:\s*(?P<val>"(?:[^\"\\]|\\.)*\\u[0-9a-fA-F]{4}(?:[^\"\\]|\\.)*")',
        flags=re.S,
    )
    arr_re = re.compile(
        r'"(?P<key>[^\"]*(?:[Cc]laim|[Ee]vid)[^\"]*)"\s*:\s*(?P<val>\[(?:[^\]]*?\\u[0-9a-fA-F]{4}[^\]]*?)\])',
        flags=re.S,
    )
    for m in str_re.finditer(raw):
        inner = m.group('val')[1:-1]
        try:
            decoded = codecs.decode(inner, 'unicode_escape')
        except Exception:
            decoded = inner.replace('\\"', '"').replace('\\\\', '\\')
        if decoded:
            escaped.add(decoded)
    for m in arr_re.finditer(raw):
        val = m.group('val')
        try:
            parsed = json.loads(val)
            for it in parsed:
                if isinstance(it, str):
                    escaped.add(it)
        except Exception:
            continue
    return escaped


def analyze_file(file_path):
    raw = open(file_path, 'r', encoding='utf-8').read()
    escaped_evidences = extract_escaped_evidences_from_raw(raw)
    data = load_json_flexible(file_path)

    rows = []
    total_traces = 0
    traces_with_literal = 0

    for si, sample in enumerate(data):
        traces = sample.get('Reasoning_traces', [])
        if isinstance(traces, str):
            traces = [traces]
        # collect sample evidence texts
        sample_evids = []
        for k, v in sample.items():
            if 'evid' in k.lower():
                if isinstance(v, list):
                    for it in v:
                        if isinstance(it, str):
                            sample_evids.append(it)
                elif isinstance(v, str):
                    sample_evids.append(v)
        for ti, tr in enumerate(traces):
            if not isinstance(tr, str):
                continue
            total_traces += 1
            has_literal = '\\u' in tr
            if has_literal:
                traces_with_literal += 1
            referenced_escaped = []
            for ev in sample_evids:
                if not ev:
                    continue
                if ev in tr and ev in escaped_evidences:
                    referenced_escaped.append(ev)
            if referenced_escaped:
                rows.append({
                    'file': file_path,
                    'sample_index': si,
                    'trace_index': ti,
                    'trace': tr,
                    'referenced_escaped_evidences': referenced_escaped,
                    'has_literal_escape': has_literal,
                })
    df = pd.DataFrame(rows)
    print(f'File: {file_path}')
    print(f'  total_traces: {total_traces}')
    print(f'  traces_with_literal_\\u: {traces_with_literal}')
    print(f'  escaped_evidence_count_in_raw: {len(escaped_evidences)}')
    if not df.empty:
        display(df)
    else:
        print('  No traces referencing escaped evidences were found.')
    return {
        'file': file_path,
        'total_traces': total_traces,
        'traces_with_literal_\\u': traces_with_literal,
        'escaped_evidence_count_in_raw': len(escaped_evidences),
        'matches_df': df,
    }

# run for all files and collect summary
summary = [analyze_file(p) for p in files]
summary_df = pd.DataFrame([{
    'file': s['file'],
    'total_traces': s['total_traces'],
    'traces_with_literal_\\u': s['traces_with_literal_\\u'],
    'escaped_evidence_count_in_raw': s['escaped_evidence_count_in_raw']
} for s in summary])
print('\nSummary:')
display(summary_df)


File: E:\Projects\Automatic Numerical Fact Verification Using Reasoning Traces\task 2\data\train.json
  total_traces: 0
  traces_with_literal_\u: 0
  escaped_evidence_count_in_raw: 1607
  No traces referencing escaped evidences were found.
File: E:\Projects\Automatic Numerical Fact Verification Using Reasoning Traces\task 2\data\validation.json
  total_traces: 24000
  traces_with_literal_\u: 0
  escaped_evidence_count_in_raw: 424
  No traces referencing escaped evidences were found.
File: E:\Projects\Automatic Numerical Fact Verification Using Reasoning Traces\task 2\data\test.json
  total_traces: 51160
  traces_with_literal_\u: 0
  escaped_evidence_count_in_raw: 9103


,file,sample_index,trace_index,trace,referenced_escaped_evidences,has_literal_escape
0,E:\Projects\Automatic Numerical Fact Verificat...,167,10,"To fact-check the claim ""National maize harves...",[National maize harvests have risen from 44 mi...,False
1,E:\Projects\Automatic Numerical Fact Verificat...,189,0,"To evaluate the claim, we will analyze the pro...",[The National Bureau of Statistics (NBS) has s...,False
2,E:\Projects\Automatic Numerical Fact Verificat...,189,1,"To fact-check the claim, we will analyze the n...",[The National Bureau of Statistics (NBS) has s...,False
3,E:\Projects\Automatic Numerical Fact Verificat...,202,1,Justification: The claim states that Nigeria's...,[The National Bureau of Statistics (NBS) says ...,False
4,E:\Projects\Automatic Numerical Fact Verificat...,296,7,"To fact-check the claim, we need to analyze bo...",[Tirupati - Kacheguda AC Double Decker Express...,False
...,...,...,...,...,...,...
116,E:\Projects\Automatic Numerical Fact Verificat...,2224,9,"To fact-check the claim, we need to analyze bo...",[President Trump is telling Republicans not to...,False
117,E:\Projects\Automatic Numerical Fact Verificat...,2260,11,"To fact-check the claim, we need to analyze th...","[As of 1990, the abortion time limit is 24 wee...",False
118,E:\Projects\Automatic Numerical Fact Verificat...,2260,14,"To fact-check the claim, we need to analyze bo...","[As of 1990, the abortion time limit is 24 wee...",False
119,E:\Projects\Automatic Numerical Fact Verificat...,2321,8,To fact-check the claim regarding Republican U...,[Thomas Massie of Kentucky exposed senior Whit...,False



Summary:


,file,total_traces,traces_with_literal_\u,escaped_evidence_count_in_raw
0,E:\Projects\Automatic Numerical Fact Verificat...,0,0,1607
1,E:\Projects\Automatic Numerical Fact Verificat...,24000,0,424
2,E:\Projects\Automatic Numerical Fact Verificat...,51160,0,9103
